# ClinicalDDI — Fingerprint Model Training

Reproduces the `fingerprint_model.onnx` used in the ClinicalDDI browser app.

**Pipeline:** DDInter CSV → balanced binary dataset → PubChem SMILES → Morgan fingerprints → MLP → ONNX export

**Output files (place in `public/` of the Next.js app):**
- `fingerprint_model.onnx` — the trained MLP classifier
- `drug_smiles.json` — drug name → SMILES mapping for the frontend

In [ ]:
# Cell 1: Install required packages
!pip install rdkit-pypi pandas numpy scikit-learn onnx skl2onnx tqdm requests -q

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import os, glob, requests, time, json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from rdkit import Chem
from rdkit.Chem import AllChem
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

np.random.seed(42)
print('All imports successful')

In [ ]:
# Cell 3: Download DDInter 2.0 CSV files
# Source: https://ddinter2.scbdd.com/download/
os.makedirs('ddinter_data', exist_ok=True)
base_url = 'https://ddinter2.scbdd.com/static/media/downloads/'
files = [
    'ddinter_downloads_code_A.csv',
    'ddinter_downloads_code_B.csv',
    'ddinter_downloads_code_D.csv',
    'ddinter_downloads_code_H.csv',
    'ddinter_downloads_code_L.csv',
    'ddinter_downloads_code_P.csv',
    'ddinter_downloads_code_R.csv',
    'ddinter_downloads_code_V.csv',
]
for fname in files:
    outpath = os.path.join('ddinter_data', fname)
    if not os.path.exists(outpath):
        r = requests.get(base_url + fname, timeout=30)
        with open(outpath, 'wb') as f:
            f.write(r.content)
        print(f'Downloaded {fname}')
    else:
        print(f'{fname} already cached')
print('DDInter CSV files ready')

In [ ]:
# Cell 4: Build balanced binary dataset
all_dfs = []
for f in glob.glob('ddinter_data/*.csv'):
    df = pd.read_csv(f)[['Drug_A', 'Drug_B']]
    df.columns = ['drug1', 'drug2']
    all_dfs.append(df)

df_pos = pd.concat(all_dfs, ignore_index=True).drop_duplicates()
df_pos['label'] = 1
print(f'Positive pairs: {len(df_pos):,}')

all_drugs = list(set(df_pos['drug1']).union(set(df_pos['drug2'])))
print(f'Unique drugs: {len(all_drugs):,}')

# Generate equal number of negative (non-interacting) pairs
pos_set = set(df_pos.apply(lambda r: tuple(sorted([r['drug1'], r['drug2']])), axis=1))
neg_pairs = set()
target = len(df_pos)
while len(neg_pairs) < target:
    d1, d2 = np.random.choice(all_drugs, 2, replace=False)
    if d1 == d2: continue
    key = tuple(sorted([d1, d2]))
    if key not in pos_set:
        neg_pairs.add(key)

df_neg = pd.DataFrame(list(neg_pairs), columns=['drug1', 'drug2'])
df_neg['label'] = 0

df_binary = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
df_binary.to_csv('binary_dataset.csv', index=False)
print(f'Balanced dataset: {len(df_binary):,} pairs')
print(df_binary['label'].value_counts())

In [ ]:
# Cell 5: Fetch SMILES from PubChem (cached)
# ~20 minutes for ~2,000 unique drugs at 0.2s/request
all_drug_names = list(set(df_binary['drug1']).union(set(df_binary['drug2'])))
smiles_map = {}

# Load existing cache if available
if os.path.exists('drug_smiles.csv'):
    df_map = pd.read_csv('drug_smiles.csv')
    smiles_map = dict(zip(df_map['drug'], df_map['smiles']))
    print(f'Loaded {len(smiles_map)} cached SMILES')

missing = [d for d in all_drug_names if d not in smiles_map]
print(f'Fetching {len(missing)} missing SMILES from PubChem...')

def get_smiles(name):
    try:
        url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/CanonicalSMILES/JSON'
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except:
        return None

for drug in tqdm(missing):
    sm = get_smiles(drug)
    if sm:
        smiles_map[drug] = sm
    time.sleep(0.2)

pd.DataFrame(list(smiles_map.items()), columns=['drug', 'smiles']).to_csv('drug_smiles.csv', index=False)
print(f'SMILES resolved: {len(smiles_map)} / {len(all_drug_names)}')

In [ ]:
# Cell 6: Compute Morgan fingerprints (radius=2, 1024 bits)
def smiles_to_fp(smiles, radius=2, nBits=1024):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
        return np.array(fp, dtype=np.float32)
    except:
        return None

drug_fp = {}
for drug, smiles in tqdm(smiles_map.items()):
    fp = smiles_to_fp(smiles)
    if fp is not None:
        drug_fp[drug] = fp
print(f'Fingerprints computed for {len(drug_fp)} drugs')

In [ ]:
# Cell 7: Build X, y arrays (concatenated fingerprint pairs)
X_list, y_list = [], []
skipped = 0
for _, row in tqdm(df_binary.iterrows(), total=len(df_binary)):
    d1, d2 = row['drug1'], row['drug2']
    if d1 in drug_fp and d2 in drug_fp:
        X_list.append(np.concatenate([drug_fp[d1], drug_fp[d2]]))
        y_list.append(row['label'])
    else:
        skipped += 1

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
print(f'Dataset: {X.shape[0]:,} pairs kept, {skipped} skipped (missing SMILES)')
print(f'Input shape: {X.shape}  — 2 x 1024 concatenated Morgan fingerprints')

In [ ]:
# Cell 8: Train / validation split (80:20, stratified)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Validation: {X_val.shape[0]:,}')

In [ ]:
# Cell 9: Train MLP (128 → 64 → 1, sigmoid)
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    batch_size=256,
    max_iter=20,
    random_state=42,
    verbose=True,
)
mlp.fit(X_train, y_train)
print('Training complete')

In [ ]:
# Cell 10: Evaluate on validation set
y_proba = mlp.predict_proba(X_val)[:, 1]
auroc   = roc_auc_score(y_val, y_proba)
acc     = accuracy_score(y_val, mlp.predict(X_val))
print(f'Validation AUROC:    {auroc:.4f}')
print(f'Validation Accuracy: {acc:.4f}')
print()
print('Target: AUROC > 0.95, Accuracy > 0.88')

In [ ]:
# Cell 11: Export to ONNX (opset 17)
initial_type = [('input', FloatTensorType([None, X.shape[1]]))]
onnx_model = convert_sklearn(mlp, initial_types=initial_type, target_opset=17)
with open('fingerprint_model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())
import os
size_mb = os.path.getsize('fingerprint_model.onnx') / 1024 / 1024
print(f'ONNX model saved: fingerprint_model.onnx  ({size_mb:.1f} MB)')

In [ ]:
# Cell 12: Save drug → SMILES JSON for the Next.js frontend
with open('drug_smiles.json', 'w') as f:
    json.dump(smiles_map, f)
print(f'drug_smiles.json saved ({len(smiles_map)} entries)')
print()
print('Done! Download these two files from the Colab file panel (left sidebar):')
print('  fingerprint_model.onnx  →  place in public/ of your Next.js app')
print('  drug_smiles.json        →  place in public/ of your Next.js app')